# Multi-antenna baseband KLT pipeline

Parametric pipeline to generate multi-antenna GUPPI raw baseband data with
`setigen`, read it back, run the KLT per frequency channel, and visualise the
results. Each generated scenario is paired with a `.meta.json` sidecar holding
the injected ground truth, so the analysis can be checked against it.

## 1. Generation

In [ ]:
import time, json
import numpy as np
import astropy.constants as const
from astropy import units as u
import setigen as stg
import katpoint

# Optional: enable GPU acceleration if CuPy is installed
import os
os.environ["SETIGEN_ENABLE_GPU"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


# ----------------------------------------------------------------------
# HELPERS: antenna construction, delays, channel -> frequency mapping
# ----------------------------------------------------------------------

def build_antennas(num_antennas, array_radius_m=1000.0, seed=42):
    """Create ``num_antennas`` katpoint antennas. Antenna 0 sits at the centre
    (reference); the others are placed at random positions inside a disk of
    radius ``array_radius_m`` (metres)."""
    rng = np.random.default_rng(seed)
    ref_ant = katpoint.Antenna("m000, -30:42:39.8, 21:26:38.0, 1035.0, 13.5")
    antennas = []
    for i in range(num_antennas):
        if i == 0:
            e = n = up = 0.0
        else:
            r = array_radius_m * np.sqrt(rng.random())   # uniform over the disk
            th = 2 * np.pi * rng.random()
            e, n, up = r * np.cos(th), r * np.sin(th), 0.0
        antennas.append(
            katpoint.Antenna(f"m{i:03d}, -30:42:39.8, 21:26:38.0, 1035.0, 13.5, {e}, {n}, {up}")
        )
    return ref_ant, antennas


def compute_delays(ref_ant, antennas, target, timestamp, sample_rate):
    """Geometric delays (in raw samples, all shifted to be positive) per antenna."""
    delays_sec = np.array([
        target.uvw(ant, timestamp, ref_ant)[2] / const.c.value
        for ant in antennas
    ])
    delays_samples = np.round(delays_sec * sample_rate).astype(int)
    delays_samples -= delays_samples.min()
    return delays_samples, delays_sec


# ----------------------------------------------------------------------
# MAIN PIPELINE
# ----------------------------------------------------------------------

def generate_scenario(
    output_stem,
    num_antennas      = 60,
    signals           = None,       # list of dicts (see examples below)
    target_radec      = ("08:35:20.6", "-45:10:35.2"),
    sample_rate       = 3e9,
    fch1_hz           = 1.4e9,
    num_branches      = 1024,
    num_taps          = 8,
    num_chans         = 64,         # coarse channels RECORDED (0..num_chans-1)
    start_chan        = 0,
    num_pols          = 2,
    noise_std         = 1.0,
    samples_per_block = 8192,       # output time frames per block
    num_blocks        = 1,
    num_subblocks     = 32,         # memory control: higher = less RAM
    array_radius_m    = 1000.0,
    seed              = 42,
    verbose           = True,
):
    """Generate a multi-antenna GUPPI raw file plus a ``.meta.json`` sidecar
    holding the injected ground truth."""
    assert samples_per_block % num_subblocks == 0, \
        "samples_per_block must be divisible by num_subblocks"
    signals = signals or []
    rng = np.random.default_rng(seed)
    chan_bw = sample_rate / num_branches

    def freq_for_channel(c, offset_frac=0.25):
        # centre of coarse channel c, offset within the channel to avoid the fine DC bin
        return fch1_hz + (start_chan + c) * chan_bw + offset_frac * chan_bw

    # --- STEP 1: antennas + delays ---
    ref_ant, antennas = build_antennas(num_antennas, array_radius_m, seed)
    target = katpoint.Target(f"target_source, radec, {target_radec[0]}, {target_radec[1]}")
    delays_samples, delays_sec = compute_delays(
        ref_ant, antennas, target, time.time(), sample_rate
    )
    if verbose:
        print(f"[{output_stem}] {num_antennas} antennas | "
              f"delay range: {delays_sec.min()*1e9:.1f} .. {delays_sec.max()*1e9:.1f} ns "
              f"({delays_samples.max()} samples)")

    # --- STEP 2: setigen array ---
    maa = stg.voltage.MultiAntennaArray(
        num_antennas = num_antennas,
        sample_rate  = sample_rate,
        fch1         = fch1_hz * u.Hz,
        ascending    = True,
        num_pols     = num_pols,
        delays       = delays_samples,
    )

    # --- STEP 3: noise (independent per antenna) ---
    for antenna in maa.antennas:
        for stream in antenna.streams:
            stream.add_noise(v_mean=0, v_std=noise_std)

    # --- STEP 3b: signals ---
    injected = []
    for k, sig in enumerate(signals):
        f_start = sig.get("f_start_hz") or freq_for_channel(sig["channel"])
        chan    = sig.get("channel")
        if chan is not None and not (0 <= chan < num_chans):
            print(f"  WARNING signal {k}: channel {chan} outside the {num_chans} "
                  f"recorded channels, it will be invisible.")
        drift = sig.get("drift_rate", 0.0) * u.Hz / u.s
        level = sig["level"]
        kind  = sig["kind"]

        if kind == "coherent":
            # sky/celestial signal: on bg_streams -> receives delays, coherent across ALL antennas
            for stream in maa.bg_streams:
                stream.add_constant_signal(f_start=f_start, drift_rate=drift, level=level)
            ants = list(range(num_antennas))

        elif kind == "per_antenna":
            # RFI / local signal: on a subset of antennas, WITHOUT sky delays
            ants = sig.get("antenna_indices")
            if ants is None:
                n = sig.get("n_antennas") or num_antennas
                ants = sorted(
                    rng.choice(num_antennas, size=min(n, num_antennas), replace=False).tolist()
                )
            for i in ants:
                for stream in maa.antennas[i].streams:
                    stream.add_constant_signal(f_start=f_start, drift_rate=drift, level=level)
        else:
            raise ValueError(f"unknown kind: {kind!r} (use 'coherent' or 'per_antenna')")

        injected.append({
            "kind": kind, "channel": chan, "f_start_hz": float(f_start),
            "drift_rate_hz_s": float(sig.get("drift_rate", 0.0)), "level": level,
            "n_antennas": len(ants), "antennas": ants,
        })
        if verbose:
            tag = "ALL" if len(ants) == num_antennas else f"{len(ants)} ant"
            print(f"  + signal {k}: {kind:11s} ch={chan} level={level} -> {tag}")

    # --- STEP 4: channelisation + writing ---
    bytes_per_sample = num_antennas * num_chans * num_pols * 2  # complex 8-bit
    block_size = bytes_per_sample * samples_per_block

    rvb = stg.voltage.RawVoltageBackend(
        maa,
        digitizer   = stg.voltage.RealQuantizer(target_fwhm=32, num_bits=8),
        filterbank  = stg.voltage.PolyphaseFilterbank(num_taps=num_taps, num_branches=num_branches),
        requantizer = stg.voltage.ComplexQuantizer(target_fwhm=32, num_bits=8),
        start_chan  = start_chan,
        num_chans   = num_chans,
        block_size  = block_size,
        blocks_per_file = 128,
        num_subblocks   = num_subblocks,
    )
    print(f"  writing ({samples_per_block} samp/block x {num_blocks} blocks)...")
    rvb.record(output_file_stem=output_stem, num_blocks=num_blocks, length_mode="num_blocks")

    # --- metadata / ground truth ---
    meta = {
        "output_stem": output_stem, "num_antennas": num_antennas,
        "num_chans": num_chans, "num_pols": num_pols, "start_chan": start_chan,
        "sample_rate": sample_rate, "fch1_hz": fch1_hz, "chan_bw": chan_bw,
        "samples_per_block": samples_per_block, "num_blocks": num_blocks,
        "signals": injected,
    }
    with open(f"{output_stem}.meta.json", "w") as f:
        json.dump(meta, f, indent=2)
    print(f"  OK wrote {output_stem}.0000.raw + {output_stem}.meta.json\n")
    return meta


## 2. Define scenarios

A signal is a dict with:

- `kind`: `"coherent"` (sky signal, goes on the bg_stream -> receives the geometric
  delays, coherent across **all** antennas) or `"per_antenna"` (RFI / local signal,
  on a subset of antennas, **without** delays).
- `channel`: recorded coarse channel to place it in (0..num_chans-1). Alternatively
  pass `f_start_hz` directly.
- `drift_rate`: Hz/s. `level`: amplitude.
- `per_antenna` only: `n_antennas` (how many, chosen at random) **or**
  `antenna_indices` (explicit list).

In [ ]:
# --- Scenario A: 60 antennas, a single coherent sky signal ---
generate_scenario(
    "scen_A", num_antennas=60,
    signals=[{"kind": "coherent", "channel": 20, "drift_rate": 15, "level": 0.05}],
)

# --- Scenario B: RFI on all antennas + coherent target (the "classic" case) ---
generate_scenario(
    "scen_B", num_antennas=60,
    signals=[
        {"kind": "per_antenna", "channel": 10, "drift_rate": 0,  "level": 0.1},   # RFI, all
        {"kind": "coherent",    "channel": 20, "drift_rate": 15, "level": 0.05},  # target
    ],
)

# --- Scenario C: signal seen by only 20 of 60 antennas (partial coverage) ---
generate_scenario(
    "scen_C", num_antennas=60,
    signals=[{"kind": "per_antenna", "channel": 30, "drift_rate": 5, "level": 0.08, "n_antennas": 20}],
)

# --- Scenario D: three signals in different channels, mixed cases ---
generate_scenario(
    "scen_D", num_antennas=80,
    signals=[
        {"kind": "per_antenna", "channel": 8,  "drift_rate": 0,  "level": 0.12, "n_antennas": 80},  # strong RFI
        {"kind": "coherent",    "channel": 25, "drift_rate": 12, "level": 0.05},                     # target A
        {"kind": "coherent",    "channel": 45, "drift_rate": -8, "level": 0.03},                     # weak target B
    ],
)


## 3. Load a scenario

In [ ]:
import json
import numpy as np
from blimpy import GuppiRaw


def load_scenario(stem, pol=0, n_samples=3072, remove_dc=True):
    """Read a generated scenario back into a (T, F, A) complex array and return
    it together with its ground-truth metadata."""
    meta = json.load(open(f"{stem}.meta.json"))
    N_A, N_CHANNELS = meta["num_antennas"], meta["num_chans"]

    raw = GuppiRaw(f"{stem}.0000.raw")
    blocks, total = [], 0
    while total < n_samples:
        try:
            _, d = raw.read_next_data_block()
        except StopIteration:
            break
        if d is None:
            break
        blocks.append(d)
        total += d.shape[1]

    raw_data = np.concatenate(blocks, axis=1)[:, :n_samples, :]      # (A*F, T, npol)
    assert raw_data.shape[0] == N_A * N_CHANNELS, \
        f"expected {N_A * N_CHANNELS} channels, found {raw_data.shape[0]}"

    # The channel axis stacks antennas: index = antenna * N_CHANNELS + channel
    raw_data = raw_data.reshape(N_A, N_CHANNELS, n_samples, raw_data.shape[2])
    data = np.transpose(raw_data[:, :, :, pol], (2, 1, 0))           # (T, F, A)
    data = np.ascontiguousarray(data, dtype=np.complex64)

    # Remove DC per (channel, antenna): kills the zero-frequency bin and the
    # common-mode quantiser DC without touching narrowband tones.
    if remove_dc:
        data = data - data.mean(axis=0, keepdims=True)

    truth = {s["channel"] for s in meta["signals"] if s["channel"] is not None}
    print(f"[{stem}] data {data.shape} | signal channels (ground truth): {sorted(truth)}")
    return data, meta


data, meta = load_scenario("scen_B", pol=0, n_samples=3072)


## 4. KLT analysis

Runs the KLT per frequency channel. The DC channel is excluded. All eigenvalues
are stored (not just the first) so the eigenspectrum of every channel can be
plotted as a heatmap later.

In [ ]:
from seti_klt import KLT, KN_rankEst
from seti_klt.utils import SimpleLogger, LogLevel

N_SAMPLES, N_CHANNELS, N_A = data.shape

EXCLUDE_CHANNELS = {0}          # DC channel; add band edges if needed, e.g. {0, 63}
channels = [c for c in range(N_CHANNELS) if c not in EXCLUDE_CHANNELS]

power_orig  = np.mean(np.abs(data)**2, axis=2)            # (T, F) -- all channels
power_recon = np.full((N_SAMPLES, N_CHANNELS), np.nan)    # NaN = not analysed

evals1_arr        = np.full(N_CHANNELS, np.nan)           # dominant eigenvalue per channel
evals_all         = np.full((N_CHANNELS, N_A), np.nan)    # full eigenspectrum per channel
K_est_per_channel = np.zeros(N_CHANNELS, dtype=int)

for freq_ch in channels:
    x_ch = data[:, freq_ch, :]
    klt_ch = KLT(SimpleLogger(level=LogLevel.INFO, console_output=False))
    klt_ch.load_signal_from_array(x_ch)
    klt_ch.apply_mrklt(n_eigenvectors=1)
    evals_ch = klt_ch.get_eigenvalues()

    evals1_arr[freq_ch] = evals_ch[0]
    evals_all[freq_ch, :len(evals_ch)] = evals_ch

    kestKN_ch, _ = KN_rankEst(ell=evals_ch, n=N_SAMPLES, beta=2, alpha=0.01)
    K_est_per_channel[freq_ch] = kestKN_ch

    if kestKN_ch > 0:
        klt_ch.apply_mrklt(n_eigenvectors=kestKN_ch)
        rec = np.abs(klt_ch.reconstructed_signal)**2      # 2D: one axis time, one antennas
        ant_axis = 0 if rec.shape[0] == N_A else 1        # reduce over the antenna axis (len == N_A)
        power_recon[:, freq_ch] = rec.mean(axis=ant_axis)
    else:
        power_recon[:, freq_ch] = 0.0                     # no coherent component -> null reconstruction

# Quick ground-truth check
truth = {s["channel"] for s in meta["signals"] if s["channel"] is not None}
for c in sorted(truth):
    print(f"channel {c}: expected signal, K_est = {K_est_per_channel[c]}")


## 5. Waterfall & PSD

In [ ]:
import matplotlib.pyplot as plt

# --- WATERFALL ---
logp = 10*np.log10(power_orig + 1e-30)
logp[:, list(EXCLUDE_CHANNELS)] = np.nan          # hide the DC channel in the Original too
vmin = float(np.nanpercentile(logp, 1))
vmax = float(np.nanpercentile(logp, 99))

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, power, title in [
    (axes[0], power_orig,  "Original"),
    (axes[1], power_recon, "Reconstructed"),
]:
    im = ax.imshow(
        10*np.log10(power + 1e-30),
        aspect="auto", cmap="viridis", origin="upper",
        vmin=vmin, vmax=vmax,
        extent=[-0.5, N_CHANNELS-0.5, N_SAMPLES-0.5, -0.5],
    )
    ax.set_xlabel("Frequency channel")
    ax.set_ylabel("Timestep")
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label="Power [dB]")
plt.tight_layout()

# --- PSD ---
psd_orig  = np.nanmean(power_orig,  axis=0)
psd_recon = np.nanmean(power_recon, axis=0)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(range(N_CHANNELS), 10*np.log10(psd_orig + 1e-30),
             color="steelblue", lw=1.5, label="Original")
axes[0].set_ylabel("Average Power [dB]")
axes[0].set_title("Power Spectral Density")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(range(N_CHANNELS), 10*np.log10(psd_orig + 1e-30),
             color="steelblue", lw=1, alpha=0.5, label="Original")
axes[1].plot(range(N_CHANNELS), 10*np.log10(psd_recon + 1e-30),
             color="crimson",   lw=2,             label="Reconstructed")
axes[1].set_xlabel("Frequency channel")
axes[1].set_ylabel("Average Power [dB]")
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()


## 6. Eigenvalue heatmap & rank per channel

Bottom panel: the full eigenspectrum of every channel. A bright bottom row
(index 0) on a channel means a dominant eigenvalue, i.e. inter-antenna
coherence there. Top panel: `K_est` per channel (how many eigenvalues were
kept). Dashed lines mark the injected ground-truth channels.

In [ ]:
# normalise each channel by its noise floor (median eigenvalue) so signals
# stand out comparably across channels with different scales
floor = np.nanmedian(evals_all, axis=1, keepdims=True)
evals_norm = evals_all / floor                       # ~1 = noise, >>1 = coherent

fig, (ax0, ax1) = plt.subplots(
    2, 1, figsize=(13, 7), sharex=True,
    gridspec_kw={"height_ratios": [1, 2]},
)

# top panel: number of eigenvalues kept per channel
ax0.bar(range(N_CHANNELS), K_est_per_channel, color="crimson", width=0.9)
ax0.set_ylabel("K kept")
ax0.set_title("Eigenvalues kept per channel (ground truth vs detected)")
ax0.grid(alpha=0.3, axis="y")
ax0.set_yticks(range(0, int(np.nanmax(K_est_per_channel)) + 2))

# bottom panel: full eigenspectrum of every channel as a heatmap
im = ax1.imshow(
    10*np.log10(evals_norm.T + 1e-12),               # dB relative to the noise floor
    aspect="auto", origin="lower", cmap="magma",
    extent=[-0.5, N_CHANNELS-0.5, -0.5, N_A-0.5],
)
ax1.set_xlabel("Frequency channel")
ax1.set_ylabel("Eigenvalue index (0 = dominant)")
plt.colorbar(im, ax=ax1, label="Eigenvalue / noise floor [dB]")

# mark ground-truth channels from the metadata
truth = {s["channel"] for s in meta["signals"] if s["channel"] is not None}
for c in truth:
    ax0.axvline(c, color="steelblue", ls="--", lw=1, alpha=0.7)
    ax1.axvline(c, color="cyan",      ls="--", lw=1, alpha=0.5)

plt.tight_layout()


## 7. Single-channel eigenspectrum (optional)

Detailed eigenspectrum for the channel with the strongest dominant eigenvalue.

In [ ]:
FREQ_CH = int(np.nanargmax(evals1_arr))
klt = KLT(SimpleLogger(level=LogLevel.INFO, console_output=False))
klt.load_signal_from_array(data[:, FREQ_CH, :])
klt.apply_mrklt(n_eigenvectors=1)
evals = klt.get_eigenvalues()
n_kept = K_est_per_channel[FREQ_CH]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.scatter(range(n_kept, len(evals)), evals[n_kept:],
           color="black", s=6,  label="Rejected")
ax.scatter(range(n_kept), evals[:n_kept],
           color="red",   s=10, label=f"Kept ({n_kept})")
ax.set_xlabel("Eigenvalue index (sorted descending)")
ax.set_ylabel("Eigenvalue magnitude")
ax.set_title(f"Eigenspectrum - channel {FREQ_CH}")
ax.legend(); ax.grid(alpha=0.3)
